In [1]:
# %pip install pymupdf python-docx
# %pip install spacy
# %pip install spacy-transformers
# For the high-accuracy Transformer model (F1 ~90%)
# %pip install https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl

In [12]:
import fitz  # PyMuPDF
import re
import os
from docx import Document
import tkinter as tk
from tkinter import filedialog
import sys

from pathlib import Path

In [3]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports: 
    Removes margin ladders (A-H), page headers, and numbers 
    while preserving structure for Legal-BERT.
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines)
    # These appear in the margins of court reporters [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)
    
    # 2. Remove Page Headers/Footers [cite: 109, 110]
    # Targeted patterns found in your specific document
    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.', 
        r'SUPREME\s*COURT\s*REPORTS',
        r'UNION\s*OF\s*INDIA\s*v\.\s*RELIANCE\s*COMMUNICATION.*',
        r'S\.\s*RAVINDRA\s*BHAT,\s*J\.'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])
    
    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])
    
    return text.strip()

In [4]:
def process_document():
    # 2. Interactive File Selection
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    # FIX 1: Define the allowed file types in a list of tuples
    file_path = filedialog.askopenfilename(
        title="Layer 1: Select Legal Document (PDF or Word)",
        filetypes=[
            ("PDF files", "*.pdf"),
            ("Word documents", "*.docx"),
            ("All files", "*.*")
        ]
    )

    if not file_path:
        print("Selection cancelled.")
        return None

    # FIX 2: Corrected the syntax for getting the extension
    # It was: os.path.splitext(file_path).[1]lower()
    ext = os.path.splitext(file_path)[1].lower() 
    
    print(f"Extracting: {os.path.basename(file_path)}...")

    # Get the file name and remove .pdf from the og filename
    fileName = str(os.path.basename(file_path)).replace(".pdf", "")

    # 3. Text Extraction
    raw_text = ""
    try:
        if ext == ".pdf":
            with fitz.open(file_path) as doc:
                raw_text = "\n".join([page.get_text("text") for page in doc])
        elif ext == ".docx":
            doc = Document(file_path)
            raw_text = "\n".join([para.text for para in doc.paragraphs])
        
        # 4. Pre-processing & Cleaning
        cleaned_text = legal_cleaning(raw_text)

        # 5. Output for later stages
        folder_name = "Text_Extracts"
        os.makedirs(folder_name, exist_ok=True)   # Creates folder if it doesn't exis   
        output_file = os.path.join(folder_name, f"{fileName}.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to: {output_file}")
        print(f"Character count: {len(cleaned_text)}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [5]:
cleaned_data = process_document() 


Extracting: 2020_1_1_7_EN.pdf...
Success! Cleaned text saved to: Text_Extracts\2020_1_1_7_EN.txt
Character count: 15220


The Segmentation method for extracting Evidence, Facts and IPCs from cleaned Text file

In [6]:
from caller import extract_legal_details

In [7]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
print(result)

{'FACTS': 'The case involves a dispute over the refund of excess amounts encashed by the Union of India from bank guarantees provided by respondent licensees (Reliance Communication Ltd/Reliance Telecom Ltd and Sistema Shyam Teleservices Ltd) for deferred spectrum charges. The respondents had successfully bid for spectrum under NIA 2013 and NIA 2015 but faced financial constraints, leading to defaults in payment. The Union encashed bank guarantees totaling Rs. 908.91 crores against an admitted liability of Rs. 774.25 crores. Although the respondents furnished fresh bank guarantees for subsequent installments, the Union refused to refund the excess amount of Rs. 134.66 crores, citing subsequent defaults. TDSAT ordered a partial refund of Rs. 104.34 crores after adjustments, which the Union challenged in the Supreme Court.', 'EVIDENCE': ['Notice Inviting Applications 2013 (NIA 2013)', 'Notice Inviting Applications 2015 (NIA 2015)', 'Scheme for Amalgamation under the Companies Act, 1956 (

In [8]:
print(f"The processed facts using gemini 3 flash are:")
case_facts = result.get('FACTS')
print(case_facts)

The processed facts using gemini 3 flash are:
The case involves a dispute over the refund of excess amounts encashed by the Union of India from bank guarantees provided by respondent licensees (Reliance Communication Ltd/Reliance Telecom Ltd and Sistema Shyam Teleservices Ltd) for deferred spectrum charges. The respondents had successfully bid for spectrum under NIA 2013 and NIA 2015 but faced financial constraints, leading to defaults in payment. The Union encashed bank guarantees totaling Rs. 908.91 crores against an admitted liability of Rs. 774.25 crores. Although the respondents furnished fresh bank guarantees for subsequent installments, the Union refused to refund the excess amount of Rs. 134.66 crores, citing subsequent defaults. TDSAT ordered a partial refund of Rs. 104.34 crores after adjustments, which the Union challenged in the Supreme Court.


In [9]:
print(f"The processed evidence using gemini 3 flash are:")
case_evidence = result.get('EVIDENCE')
print(case_evidence)

The processed evidence using gemini 3 flash are:
['Notice Inviting Applications 2013 (NIA 2013)', 'Notice Inviting Applications 2015 (NIA 2015)', 'Scheme for Amalgamation under the Companies Act, 1956 (Sistema merger with RCL)', 'Union approval order dated 20.10.2017 regarding spectrum transfer', 'Modified payment term letters dated 19.03.2018', 'Bank Guarantees amounting to Rs. 908.91 crores, Rs. 774.25 crores, and Rs. 390.41 crores', 'RBI guidelines for Strategic Debt Restructuring (SDR)', 'TDSAT judgment and order dated 21.12.2018 in Telecom Petition No. 196 of 2018', 'Orders of the NCLAT regarding stay on encashment for subsequent defaults']


In [10]:
print(f"The processed IPCs using gemini 3 flash are:")
case_ipc = result.get('IPC/STATUTES')
print(case_ipc)

The processed IPCs using gemini 3 flash are:
['Companies Act, 1956', 'Insolvency and Bankruptcy Code (IBC)', 'Clause 4.5b(ix) of NIA 2013', 'Clause 4.5b(x) of NIA 2015', 'Clause 13.1 of the License Agreement', 'Clause 13.2 of the License Agreement']


Call the Evidence Scrutinization Llama 3 model from HF

In [ ]:
# Use getcwd() instead of __file__
current_dir = os.getcwd()
target_path = os.path.abspath(os.path.join(current_dir, "..", "2_Evidence_Scrutinization"))

if target_path not in sys.path:
    sys.path.append(target_path)

from scrutiny import scrutinize_legal_data

ModuleNotFoundError: No module named 'scrutiny'

In [ ]:
from scrutiny import scrutinize_legal_data

In [ ]:
scr_result = scrutinize_legal_data(case_facts, case_evidence, case_ipc)